# 08. MCP 고급 서버 활용

> 내가 만든 MCP 서버 외에도 **Context7, Smithery 같은 3rd-party 레지스트리**를 그대로 가져다 쓸 수 있어요. 외부 MCP 서버를 조합해 복합 워크플로우를 짜봐요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **3rd Party MCP 서버**(Context7)를 연결하여 최신 기술 문서를 검색할 수 있어요
2. **복수의 MCP 서버**를 동시에 연결하고 각 서버의 도구를 통합 관리할 수 있어요
3. **MCP 도구와 Tavily 웹 검색**을 결합한 복합 워크플로우를 구축할 수 있어요
4. **Smithery AI** 레지스트리를 활용하여 다양한 3rd Party MCP 서버를 탐색할 수 있어요
5. **복합 작업**(문서 검색 → 웹 검색 → 코드 생성)을 자동화하는 에이전트를 만들 수 있어요

## 사전 지식

- `06-MCP-Server-Basics.ipynb`: FastMCP 서버 생성, stdio/HTTP 전송 방식
- `07-MCP-Agent-Integration.ipynb`: MultiServerMCPClient, create_agent와 MCP 통합
- LangGraph StateGraph 기본 구조 (Part 2 참조)

## 개념 설명: 3rd Party MCP 서버 생태계

MCP(Model Context Protocol)의 가장 강력한 특징은 **서드파티 생태계**예요. 누구나 MCP 서버를 만들고 공유할 수 있고, 이미 수백 개의 유용한 서버들이 공개되어 있어요.

| 서버 | 제공 기능 | 설치 방식 |
|------|----------|----------|
| **Context7** | 라이브러리 최신 문서 검색 | `npx @upstash/context7-mcp` |
| **Brave Search** | 웹 검색 | Smithery 레지스트리 |
| **GitHub** | 저장소 관리 | `npx @modelcontextprotocol/server-github` |
| **Filesystem** | 파일 시스템 접근 | `npx @modelcontextprotocol/server-filesystem` |
| **PostgreSQL** | DB 조회 | `npx @modelcontextprotocol/server-postgres` |

> 🔑 **핵심 개념**: MCP 서버는 **표준 인터페이스**로 동작해요. 마치 스마트폰 **앱 스토어**처럼, Smithery AI에서 원하는 MCP 서버를 찾아 한 줄 명령으로 설치하면 바로 에이전트에서 사용할 수 있어요. 서버가 `stdio`나 `HTTP`로 통신하기만 하면 언어, 프레임워크 무관하게 에이전트에 연결할 수 있어요.

### Smithery AI 레지스트리

[Smithery AI](https://smithery.ai/)는 MCP 서버를 검색하고 설치할 수 있는 공식 레지스트리예요. npm 패키지처럼 서버를 검색하고, 설명과 도구 목록을 확인한 후 바로 사용할 수 있어요.

```bash
# Smithery CLI 설치
npx @smithery/cli install @upstash/context7-mcp --client claude
```

> 💡 **실무 팁**: 새로운 기능이 필요할 때 직접 MCP 서버를 만들기 전에 Smithery에서 검색해보세요. 대부분의 일반적인 도구들은 이미 공개된 서버가 있어요.

## 전체 아키텍처

```mermaid
flowchart TD
    User([사용자 질문]) --> Agent[에이전트<br>create_agent]
    Agent --> Client[MultiServerMCPClient]
    Client --> S1[날씨 서버<br>mcp_server_local.py<br>stdio]
    Client --> S2[시간 서버<br>mcp_server_remote.py<br>HTTP]
    Client --> S3["Context7 MCP<br>@upstash/context7-mcp<br>npx + stdio"]
    Client --> S4[웹 검색 서버<br>06_web_search_server.py<br>stdio]
    S1 -- get_weather --> Tools[통합 도구 목록]
    S2 -- get_current_time --> Tools
    S3 -- resolve-library-id<br>query-docs --> Tools
    S4 -- web_search<br>search_news --> Tools
    Tools --> Agent
    Agent --> Result([최종 응답])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class User input
    class Agent,Client process
    class S1,S2,S3,S4 storage
    class Tools,Result output
```

> 🎯 **강의 포인트**: 에이전트 입장에서는 도구가 어느 서버에서 왔는지 알 필요가 없어요. `MultiServerMCPClient`가 모든 서버의 도구를 **하나의 목록**으로 통합해주기 때문이에요.

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 및 추적 설정
# ---------------------------------------------------
import os
from dotenv import load_dotenv

# .env 파일에서 API 키들을 로드해요
# OPENAI_API_KEY, TAVILY_API_KEY 등이 필요해요
load_dotenv()

# LangSmith 추적 설정 (선택사항)
# 에이전트의 실행 과정을 LangSmith에서 확인할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-MCP-Advanced"

# 환경 설정 완료

True

In [2]:
# ---------------------------------------------------
# 필수 라이브러리 import
# ---------------------------------------------------
import asyncio
import concurrent.futures
import uuid

# LangChain V1 API 사용
from langchain.agents import create_agent        # V1 에이전트 생성 함수
from langchain.chat_models import init_chat_model # 모델 초기화
from langchain_core.runnables import RunnableConfig

# LangGraph 체크포인터: create_agent 실행 상태 저장에 사용해요
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 여러 MCP 서버를 하나로 관리해요
from langchain_mcp_adapters.client import MultiServerMCPClient


def run_async(coro):
    """MCP 비동기 코루틴을 안전하게 실행해요.
    
    Python 3.14 + anyio 환경에서 nest_asyncio를 사용하면
    asyncio.current_task()가 None을 반환하는 호환성 문제가 있어요.
    별도 스레드에서 새 이벤트 루프로 실행하면 이 문제를 우회할 수 있어요.
    
    Args:
        coro: 실행할 코루틴
        
    Returns:
        코루틴 실행 결과
    """
    # 별도 스레드에서 독립적인 이벤트 루프로 실행해요
    # anyio의 TaskGroup이 asyncio.current_task()를 필요로 하는데
    # nest_asyncio 환경에서 None을 반환하는 문제를 우회해요
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(asyncio.run, coro)
        return future.result()


# 라이브러리 로드 완료

## 1. 헬퍼 함수 정의

이 노트북 전체에서 사용할 공통 헬퍼 함수들을 먼저 정의할게요.

> 🔑 **핵심 개념**: MCP 클라이언트는 내부적으로 `anyio` 비동기 라이브러리를 사용해요. Jupyter 환경에서는 `nest_asyncio`와 `anyio`의 호환성 문제가 있어서, 위에서 정의한 `run_async()` 헬퍼를 통해 안전하게 실행해요. 프로덕션 코드에서는 `async def main()` 함수 안에서 `await`를 직접 사용하는 것이 일반적이에요.

In [3]:
# ---------------------------------------------------
# 공통 헬퍼 함수
# ---------------------------------------------------

async def setup_mcp_tools(server_configs: dict) -> list:
    """MCP 서버들에 연결하고 도구 목록을 반환해요.
    
    Args:
        server_configs: MCP 서버 구성 딕셔너리
        
    Returns:
        로드된 도구 목록
    """
    # MultiServerMCPClient로 여러 서버를 동시에 관리해요
    client = MultiServerMCPClient(server_configs)
    
    # 모든 서버에서 도구를 가져와 통합해요
    tools = await client.get_tools()
    
    # 로드된 도구 목록을 출력해요
    print(f"[MCP] 총 {len(tools)}개의 도구가 로드되었어요:")
    for tool in tools:
        print(f"  - {tool.name}: {tool.description[:60]}..." 
              if len(tool.description) > 60 
              else f"  - {tool.name}: {tool.description}")
    
    return tools


async def run_agent_stream(agent, user_message: str, config: RunnableConfig) -> None:
    """에이전트를 스트리밍 모드로 실행하고 결과를 출력해요.
    
    그래프 스트리밍 실행 결과를 출력하는 함수예요.
    
    Args:
        agent: 실행할 에이전트 (컴파일된 그래프)
        user_message: 사용자 입력 메시지
        config: RunnableConfig (thread_id 등 포함)
    """
    inputs = {"messages": [("human", user_message)]}
    
    # stream_mode="updates"로 각 노드의 업데이트를 실시간으로 받아요
    async for chunk in agent.astream(inputs, config, stream_mode="updates"):
        for node_name, node_output in chunk.items():
            print(f"\n{'='*50}")
            print(f"노드: {node_name}")
            print(f"{'='*50}")
            
            if "messages" in node_output:
                # 노드가 생성한 메시지를 출력해요
                for msg in node_output["messages"]:
                    msg.pretty_print()


def make_config(thread_id: str = None) -> RunnableConfig:
    """대화 스레드 설정을 생성해요.
    
    Args:
        thread_id: 스레드 ID (없으면 자동 생성)
        
    Returns:
        RunnableConfig 객체
    """
    # UUID로 고유한 스레드 ID를 생성해요
    tid = thread_id or str(uuid.uuid4())
    return RunnableConfig(configurable={"thread_id": tid})


# 헬퍼 함수 정의 완료

## 2. 기본 모델 설정

이 노트북에서 사용할 기본 LLM을 초기화해요.

> 💡 **실무 팁**: `temperature=0`으로 설정하면 결과가 일관성 있게 나와요. 특히 도구 호출을 해야 하는 에이전트에서는 낮은 temperature를 권장해요.

In [4]:
# ---------------------------------------------------
# LLM 초기화
# ---------------------------------------------------
# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 필요에 따라 다른 모델로 변경할 수 있어요:
#   - "openai:gpt-4o": 더 강력한 추론이 필요할 때
#   - "anthropic:claude-sonnet-4-5": Anthropic 모델 사용 시
#   - "ollama:llama3": 로컬에서 실행할 때 (무료)
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

print(f"모델 초기화 완료: gpt-4o-mini")

모델 초기화 완료: gpt-4o-mini


## 3. Context7 MCP 서버 연결

[Context7](https://github.com/upstash/context7)은 **최신 라이브러리 공식 문서**를 실시간으로 검색할 수 있는 MCP 서버예요.

LLM의 학습 데이터 한계를 극복하는 데 유용해요. 예를 들어:
- LangGraph V1의 최신 API 사용법
- React 19의 변경사항

| Context7 도구 | 용도 |
|--------------|------|
| `resolve-library-id` | 라이브러리 이름으로 ID를 검색해요 |
| `query-docs` | 특정 라이브러리의 문서를 검색해요 |

> ⚠️ **자주 하는 실수**: Context7를 사용하려면 `Node.js`와 `npx`가 설치되어 있어야 해요. `npx --version`으로 확인하세요. 없으면 `brew install node` (macOS) 또는 공식 Node.js 사이트에서 설치하세요.

> 🎯 **강의 포인트**: `npx -y @upstash/context7-mcp@latest`는 npm 패키지를 설치 없이 바로 실행해요. MCP 서버를 설치하지 않고도 `npx`로 바로 사용할 수 있다는 점이 MCP 생태계의 편의성이에요.

In [5]:
# ---------------------------------------------------
# npx 설치 확인
# ---------------------------------------------------
import subprocess

# Node.js와 npx가 설치되어 있는지 확인해요
try:
    result = subprocess.run(
        ["npx", "--version"], 
        capture_output=True, 
        text=True, 
        timeout=10
    )
    print(f"npx 버전: {result.stdout.strip()}")
    # Context7 MCP 서버 사용 가능해요
except FileNotFoundError:
    # 경고: npx가 설치되어 있지 않아요.
    # Node.js 설치 방법:
    #   macOS: brew install node
    #   Ubuntu: sudo apt install nodejs npm
    #   Windows: https://nodejs.org에서 설치
    pass

npx 버전: 11.12.1


In [6]:
# ---------------------------------------------------
# Context7 MCP 서버 연결 및 도구 로드
# ---------------------------------------------------
# Context7 서버만 먼저 연결해서 어떤 도구들이 있는지 확인해요
context7_config = {
    "context7-mcp": {
        "command": "npx",
        # -y 플래그: 설치 확인 프롬프트를 자동으로 수락해요
        # @latest: 항상 최신 버전을 사용해요
        "args": ["-y", "@upstash/context7-mcp@latest"],
        "transport": "stdio",  # stdio 방식으로 통신해요
    },
}

# Context7 도구를 로드해요 (run_async로 anyio 기반 비동기 함수 실행)
context7_tools = run_async(setup_mcp_tools(context7_config))

[MCP] 총 2개의 도구가 로드되었어요:
  - resolve-library-id: Resolves a package/product name to a Context7-compatible lib...
  - query-docs: Retrieves and queries up-to-date documentation and code exam...


In [7]:
# ---------------------------------------------------
# Context7 에이전트 생성 및 문서 검색 테스트
# ---------------------------------------------------
# Context7 도구만 사용하는 에이전트를 만들어요
context7_agent = create_agent(
    llm,
    context7_tools,
    checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장해요
)

# 새로운 대화 스레드 설정
config = make_config()

# LangGraph 최신 문서에서 StateGraph 사용법 검색
# LangGraph 문서 검색 중...
run_async(run_agent_stream(
    context7_agent,
    "LangGraph의 StateGraph를 사용하는 방법을 공식 문서에서 찾아서 설명해줘",
    config,
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  resolve-library-id (call_M3x5TCRKvpitIBwjuRP9bLbG)
 Call ID: call_M3x5TCRKvpitIBwjuRP9bLbG
  Args:
    query: LangGraph StateGraph 사용법
    libraryName: LangGraph

노드: tools
================================= Tool Message =================================
Name: resolve-library-id

[{'type': 'text', 'text': 'Available Libraries:\n\n- Title: LangGraph\n- Context7-compatible library ID: /websites/langchain_oss_python_langgraph\n- Description: LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents, offering durable execution, human-in-the-loop capabilities, and comprehensive memory.\n- Code Snippets: 1023\n- Source Reputation: High\n- Benchmark Score: 82.6\n----------\n- Title: LangGraph\n- Context7-compatible library ID: /langchain-ai/langgraph\n- Description: Build resilient language agents as graphs.\n- Code Sni

## 4. 다중 MCP 서버 통합 (3개 서버)

이제 **날씨 서버(stdio) + 시간 서버(HTTP) + Context7(npx)** 세 개를 동시에 연결해볼게요.

```mermaid
flowchart LR
    Client[MultiServerMCPClient] --> W[날씨 서버<br>stdio]
    Client --> T[시간 서버<br>HTTP:8002]
    Client --> C[Context7<br>npx stdio]
    W -- get_weather --> M[통합 도구 4개]
    T -- get_current_time --> M
    C -- resolve-library-id<br>query-docs --> M

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class Client process
    class W,T,C storage
    class M output
```

> 🔑 **핵심 개념**: 서버마다 전송 방식이 다를 수 있어요. `stdio`(로컬 프로세스), `streamable_http`(원격 HTTP), `npx`(Node.js 패키지) 모두 동일한 `MultiServerMCPClient`로 관리돼요.

> ⚠️ **자주 하는 실수**: HTTP 방식의 시간 서버는 미리 실행해두어야 해요. 별도 터미널에서 `uv run python servers/02_time_server.py`를 실행하세요.

In [8]:
# ---------------------------------------------------
# 3개 서버 통합 구성
# ---------------------------------------------------
# 각 서버의 연결 방식이 다르지만 동일한 딕셔너리에 정의해요
# ⚠️ 주의: current_time 서버(HTTP)는 사전에 별도 터미널에서 실행이 필요해요:
#    uv run python servers/02_time_server.py

multi_server_configs = {
    # 1. 날씨 서버: Python 스크립트를 stdio로 실행해요
    "weather": {
        "command": "uv",
        "args": ["run", "python", "servers/01_weather_server.py"],
        "transport": "stdio",
    },
    # 2. 시간 서버: 미리 실행된 HTTP 서버에 연결해요
    #    별도 터미널에서 사전에 실행해두세요:
    #    uv run python servers/02_time_server.py
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",
        "transport": "streamable_http",  # HTTP 스트리밍 방식
    },
    # 3. Context7: npx로 Node.js MCP 서버를 실행해요
    "context7-mcp": {
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp@latest"],
        "transport": "stdio",
    },
}

# HTTP 서버 연결 실패를 대비해서 예외 처리를 적용해요
# (시간 서버 미실행 환경에서도 나머지 서버가 동작할 수 있어요)
try:
    multi_tools = run_async(setup_mcp_tools(multi_server_configs))
    print(f"\n총 {len(multi_tools)}개 도구를 3개 서버에서 로드했어요")
except Exception as e:
    print(f"경고: 3개 서버 통합 로드 실패 - {type(e).__name__}: {str(e)[:100]}")
    # 시간 서버(HTTP)가 실행 중이지 않을 수 있어요.
    # 별도 터미널에서 실행: uv run python servers/02_time_server.py
    # Context7 + 날씨 서버(2개)로만 계속 진행해요...
    
    # HTTP 서버 없이 stdio 서버만으로 재시도해요
    fallback_configs = {
        "weather": {
            "command": "uv",
            "args": ["run", "python", "servers/01_weather_server.py"],
            "transport": "stdio",
        },
        "context7-mcp": {
            "command": "npx",
            "args": ["-y", "@upstash/context7-mcp@latest"],
            "transport": "stdio",
        },
    }
    multi_tools = run_async(setup_mcp_tools(fallback_configs))
    print(f"\n{len(multi_tools)}개 도구를 2개 서버(날씨 + Context7)에서 로드했어요")

[MCP] 총 4개의 도구가 로드되었어요:
  - get_weather: 지정한 도시의 현재 날씨를 조회해요.

Args:
    location: 날씨를 조회할 도시 이름 (예: ...
  - get_current_time: 지정한 시간대의 현재 시간을 조회해요.

Args:
    timezone: 조회할 시간대 (기본값: Asi...
  - resolve-library-id: Resolves a package/product name to a Context7-compatible lib...
  - query-docs: Retrieves and queries up-to-date documentation and code exam...

총 4개 도구를 3개 서버에서 로드했어요


In [9]:
# ---------------------------------------------------
# 3개 서버 통합 에이전트 생성
# ---------------------------------------------------
# 4개 도구를 모두 사용할 수 있는 에이전트를 생성해요
multi_agent = create_agent(
    llm,
    multi_tools,
    checkpointer=InMemorySaver(),
)

# 대화 스레드 설정 (같은 thread_id면 대화가 이어져요)
config = make_config()

# 첫 번째 질문: 시간 서버 사용
# [질문 1] 현재 시간 조회
run_async(run_agent_stream(
    multi_agent,
    "현재 시간을 알려주세요",
    config,
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_Fb4ld5WGe67bOgQYWuTYbeeG)
 Call ID: call_Fb4ld5WGe67bOgQYWuTYbeeG
  Args:

노드: tools
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '현재 Asia/Seoul 시간: 2026-06-01 16:34:16 KST', 'id': 'lc_c09ce857-b9b5-4fb2-83b8-7b31379c3c7c'}]

노드: model
================================== Ai Message ==================================

현재 시간은 2026년 6월 1일 16시 34분 16초 (KST)입니다.


In [10]:
# ---------------------------------------------------
# 같은 대화에서 다른 서버의 도구 사용
# ---------------------------------------------------
# 같은 config(thread_id)를 사용하면 이전 대화가 이어져요
# [질문 2] 날씨 조회 (같은 대화 스레드)
run_async(run_agent_stream(
    multi_agent,
    "서울의 날씨도 알려주세요",
    config,  # 같은 thread_id를 사용해요
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_06rIqrn41SZCBrrLRqCm4vpK)
 Call ID: call_06rIqrn41SZCBrrLRqCm4vpK
  Args:
    location: 서울

노드: tools
================================= Tool Message =================================
Name: get_weather

[{'type': 'text', 'text': "It's always Sunny in 서울", 'id': 'lc_5a8d7f8f-b9a4-4c59-b962-b113635934d2'}]

노드: model
================================== Ai Message ==================================

서울의 날씨는 맑습니다.


## 5. 커스텀 워크플로우는 07장에서만 깊게 다룬다

`StateGraph + ToolNode`로 MCP 도구 루프를 직접 만드는 구현은 **05_Agent_Development/07-MCP-Agent-Integration.ipynb**가 소유해요. 이 노트북은 같은 루프를 다시 작성하지 않고, 고급 MCP 서버를 붙일 때 달라지는 **서버 구성, 도구 조합, 도구 큐레이션**에 집중해요.

| 구분 | 담당 범위 |
|------|-----------|
| 07-MCP-Agent-Integration | `ToolNode`, `tools_condition`, `StateGraph`로 MCP 루프 직접 구성 |
| 08-MCP-Advanced-Servers | Context7, 웹 검색, Smithery, 다중 서버, 도구 수 관리 |

> 🔁 **짧은 연결**: 07장의 커스텀 그래프가 필요하면 `ToolNode(tools)` 자리에 여기서 로드한 `context7_tools`, `multi_tools`, `curated_tools`를 넘기면 돼요. 이 장에서는 그 구현을 반복하지 않고, 아래부터 `create_agent` 기반으로 고급 MCP 서버 활용 흐름을 이어갑니다.


## 6. 복합 작업: 최신 문서 검색 + 코드 생성

Context7으로 **최신 LangGraph 문서를 검색**하고, 그 결과를 바탕으로 **코드를 생성**하는 복합 작업을 실행해볼게요.

이것이 MCP의 핵심 사용 사례예요:

```
사용자: "LangGraph로 ReAct 에이전트 만들어줘"
         ↓
에이전트: resolve-library-id("langgraph")  ← Context7
         ↓
에이전트: query-docs(library_id, "react agent")  ← Context7
         ↓
에이전트: [최신 문서 기반으로] 코드 생성
```

> 🎯 **강의 포인트**: LLM의 학습 데이터에는 없는 **최신 API**도 Context7를 통해 정확하게 생성할 수 있어요. 이것이 RAG(검색 증강 생성)를 MCP로 구현한 예시예요.

In [11]:
# ---------------------------------------------------
# 복합 작업: 최신 문서 → 코드 생성
# ---------------------------------------------------
# Context7 전용 create_agent 그래프 사용
config = make_config()

# 복합 작업 실행: LangGraph 최신 문서 검색 → 에이전트 코드 생성
# (Context7이 LangGraph 공식 문서를 실시간으로 검색해요)

run_async(run_agent_stream(
    context7_agent,
    "최신 LangGraph 공식 문서에서 create_agent 사용법을 검색하고, "
    "검색된 내용을 바탕으로 날씨를 알려주는 간단한 에이전트 코드를 작성해줘",
    config,
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  resolve-library-id (call_mnq49eiZYYrvzJ92CSZx3MDK)
 Call ID: call_mnq49eiZYYrvzJ92CSZx3MDK
  Args:
    query: create_agent 사용법
    libraryName: LangGraph

노드: tools
================================= Tool Message =================================
Name: resolve-library-id

[{'type': 'text', 'text': 'Available Libraries:\n\n- Title: LangGraph\n- Context7-compatible library ID: /websites/langchain_oss_python_langgraph\n- Description: LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents, offering durable execution, human-in-the-loop capabilities, and comprehensive memory.\n- Code Snippets: 1023\n- Source Reputation: High\n- Benchmark Score: 82.6\n----------\n- Title: LangGraph\n- Context7-compatible library ID: /langchain-ai/langgraph\n- Description: Build resilient language agents as graphs.\n- Code Snippets: 5

## 7. Tavily 웹 검색 + MCP 결합

MCP 도구와 일반 LangChain 도구를 **함께** 사용할 수 있어요. MCP에서 로드한 도구와 Tavily 웹 검색 도구를 합쳐서 하나의 에이전트에 제공할게요.

> 💡 **실무 팁**: `client.get_tools()`가 반환하는 목록은 일반 LangChain `BaseTool` 객체예요. Tavily나 직접 만든 `@tool` 데코레이터 함수와 함께 리스트에 합쳐서 사용할 수 있어요.

> ⚠️ **자주 하는 실수**: Tavily를 사용하려면 `TAVILY_API_KEY`가 `.env`에 있어야 해요. 없으면 `KeyError`가 발생해요.

In [12]:
# ---------------------------------------------------
# MCP 도구 + Tavily 웹 검색 도구 통합
# ---------------------------------------------------
from langchain_tavily import TavilySearch

# Tavily 웹 검색 도구 생성
# max_results: 반환할 검색 결과 수를 제한해요
tavily_tool = TavilySearch(max_results=3)

# Context7 MCP 도구와 Tavily 도구를 합쳐요
# MCP 도구는 일반 LangChain 도구와 동일하게 리스트에 추가할 수 있어요
combined_tools = context7_tools + [tavily_tool]

print(f"통합 도구 목록 ({len(combined_tools)}개):")
for t in combined_tools:
    print(f"  - {t.name}")

통합 도구 목록 (3개):
  - resolve-library-id
  - query-docs
  - tavily_search


In [13]:
# ---------------------------------------------------
# 통합 도구를 사용하는 에이전트 생성
# ---------------------------------------------------
# MCP 도구(Context7) + Tavily 웹 검색을 함께 사용해요
combined_agent = create_agent(
    llm,
    combined_tools,
    checkpointer=InMemorySaver(),
)

config = make_config()

# 복합 작업: 공식 문서 검색 + 최신 뉴스 검색 + 코드 생성
# 복합 작업: Context7 문서 검색 + Tavily 뉴스 검색
run_async(run_agent_stream(
    combined_agent,
    "1) LangGraph 공식 문서에서 StateGraph 기본 사용법을 검색하고, "
    "2) Tavily로 최신 LangGraph 관련 뉴스도 검색해서, "
    "두 결과를 종합해서 알려줘",
    config,
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  resolve-library-id (call_kbbRWT9cULhZsukiY7cVynji)
 Call ID: call_kbbRWT9cULhZsukiY7cVynji
  Args:
    query: StateGraph 기본 사용법
    libraryName: LangGraph

노드: tools
================================= Tool Message =================================
Name: resolve-library-id

[{'type': 'text', 'text': 'Available Libraries:\n\n- Title: LangGraph\n- Context7-compatible library ID: /websites/langchain_oss_python_langgraph\n- Description: LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents, offering durable execution, human-in-the-loop capabilities, and comprehensive memory.\n- Code Snippets: 1023\n- Source Reputation: High\n- Benchmark Score: 82.6\n----------\n- Title: LangGraph\n- Context7-compatible library ID: /langchain-ai/langgraph\n- Description: Build resilient language agents as graphs.\n- Code Snippets: 

## 8. 실습: 웹 검색 MCP 서버 완성하기

`servers/06_web_search_server.py`를 살펴보고 직접 사용해볼게요.

이 서버는 두 개의 도구를 제공해요:
- `web_search`: 일반 웹 검색
- `search_news`: 뉴스 특화 검색

> 🎯 **강의 포인트**: 실습 파일(`06_web_search_server.py`)은 완성된 코드예요. 어떻게 FastMCP로 Tavily를 감싸서 MCP 서버로 만드는지 패턴을 익혀보세요.

In [14]:
# ---------------------------------------------------
# 웹 검색 MCP 서버 연결
# ---------------------------------------------------
# 실습 파일로 만든 웹 검색 서버를 MCP로 연결해요
web_search_config = {
    "web_search": {
        "command": "uv",
        "args": ["run", "python", "servers/06_web_search_server.py"],
        "transport": "stdio",
    },
}

# 웹 검색 도구 로드
web_search_tools = run_async(setup_mcp_tools(web_search_config))

[MCP] 총 2개의 도구가 로드되었어요:
  - web_search: Tavily API를 사용하여 실시간 웹 검색을 수행해요.

인터넷에서 최신 정보를 검색하고 결과를 요약하여...
  - search_news: 특정 주제에 대한 최신 뉴스를 검색해요.

지정된 주제에 대해 최근 며칠 이내의 뉴스를 검색하고
가장 관련성...


In [15]:
# ---------------------------------------------------
# 웹 검색 에이전트 테스트
# ---------------------------------------------------
web_search_agent = create_agent(
    llm,
    web_search_tools,
    checkpointer=InMemorySaver(),
)

config = make_config()

# 웹 검색 서버 테스트
run_async(run_agent_stream(
    web_search_agent,
    "Python 3.13의 새로운 기능을 검색해줘",
    config,
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  web_search (call_OwgTmVYHXUsCVBnDPf3eVe64)
 Call ID: call_OwgTmVYHXUsCVBnDPf3eVe64
  Args:
    query: Python 3.13 new features

노드: tools
================================= Tool Message =================================
Name: web_search

[{'type': 'text', 'text': "[웹 검색 결과: 'Python 3.13 new features']\n", 'id': 'lc_42509b50-70ea-439b-810b-da6aac53261e'}]

노드: model
================================== Ai Message ==================================
Tool Calls:
  search_news (call_n9YOp2vn0nWHnJaGpwYD3qG5)
 Call ID: call_n9YOp2vn0nWHnJaGpwYD3qG5
  Args:
    topic: Python 3.13

노드: tools
================================= Tool Message =================================
Name: search_news

[{'type': 'text', 'text': "'Python 3.13'에 대한 최근 뉴스를 찾을 수 없어요.", 'id': 'lc_e53dcdd5-27a0-4ece-9803-bf231aed9658'}]

노드: model
================================== Ai Message ==================================


## 실습 해설: 3개 서버 통합 에이전트

아래 완성 코드는 **Context7 + 웹검색 + 시간 서버**를 하나의 에이전트에 연결하는 예제예요.

In [16]:
# ============================================================
# 실습 해설: 3개 서버(Context7 + 웹검색 + 시간)를 통합한
#        에이전트를 만들고 다음 질문을 실행해요:
#        "LangGraph 최신 문서를 검색하고,
#         관련 뉴스도 찾아서,
#         현재 시간 기준으로 종합 리포트를 작성해줘"
#
# 예상 결과: 에이전트가 공식 문서 검색, 뉴스 검색, 현재 시간 조회를
#             조합해 종합적인 리포트를 제공해요.
# ============================================================

todo_server_configs = {
    "context7-mcp": {
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp@latest"],
        "transport": "stdio",
    },
    "web_search": {
        "command": "uv",
        "args": ["run", "python", "servers/06_web_search_server.py"],
        "transport": "stdio",
    },
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",
        "transport": "streamable_http",
    },
}

try:
    todo_tools = run_async(setup_mcp_tools(todo_server_configs))
except Exception as e:
    print(f"경고: 시간 서버 연결 실패 - {type(e).__name__}: {str(e)[:100]}")
    fallback_todo_configs = {
        name: config
        for name, config in todo_server_configs.items()
        if name != "current_time"
    }
    todo_tools = run_async(setup_mcp_tools(fallback_todo_configs))

print(f"실습 에이전트 도구 수: {len(todo_tools)}개")

todo_agent = create_agent(llm, todo_tools, checkpointer=InMemorySaver())

config = make_config()
run_async(run_agent_stream(
    todo_agent,
    "LangGraph 최신 문서를 검색하고, 관련 뉴스도 찾아서, "
    "가능하면 현재 시간 기준으로 종합 리포트를 작성해줘",
    config,
))

[MCP] 총 5개의 도구가 로드되었어요:
  - resolve-library-id: Resolves a package/product name to a Context7-compatible lib...
  - query-docs: Retrieves and queries up-to-date documentation and code exam...
  - web_search: Tavily API를 사용하여 실시간 웹 검색을 수행해요.

인터넷에서 최신 정보를 검색하고 결과를 요약하여...
  - search_news: 특정 주제에 대한 최신 뉴스를 검색해요.

지정된 주제에 대해 최근 며칠 이내의 뉴스를 검색하고
가장 관련성...
  - get_current_time: 지정한 시간대의 현재 시간을 조회해요.

Args:
    timezone: 조회할 시간대 (기본값: Asi...
실습 에이전트 도구 수: 5개

노드: model
================================== Ai Message ==================================
Tool Calls:
  resolve-library-id (call_MvH0yqKe2JmoikLM2Eqk1m3d)
 Call ID: call_MvH0yqKe2JmoikLM2Eqk1m3d
  Args:
    query: LangGraph
    libraryName: LangGraph
  search_news (call_PIDPHXcP581uPqvnzAxmktj4)
 Call ID: call_PIDPHXcP581uPqvnzAxmktj4
  Args:
    topic: LangGraph
    days_back: 7
  get_current_time (call_y5MEgeTy7Tjl4k3h3VN7eQvz)
 Call ID: call_y5MEgeTy7Tjl4k3h3VN7eQvz
  Args:

노드: tools
================================= Tool Message ==

## 9. Smithery AI 레지스트리 소개

[Smithery AI](https://smithery.ai/)는 MCP 서버를 검색하고 설치할 수 있는 **공식 레지스트리**예요.

### 활용 방법

1. **검색**: https://smithery.ai 에서 원하는 기능 검색
2. **설치 명령 확인**: 각 서버 페이지에서 `npx` 명령을 확인
3. **server_configs에 추가**: 아래와 같이 바로 사용

```python
# Smithery에서 찾은 서버를 바로 추가할 수 있어요
server_configs = {
    # Brave 웹 검색 MCP 서버 (Smithery 레지스트리)
    "brave-search": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-brave-search"],
        "transport": "stdio",
        "env": {"BRAVE_API_KEY": "your-brave-api-key"}
    }
}
```

> 💡 **실무 팁**: MCP 서버를 직접 만들 필요 없이 Smithery에서 검색하면 대부분의 일반적인 도구들을 바로 사용할 수 있어요. GitHub, Slack, Notion, PostgreSQL, Google Drive 등 수백 개의 서버가 공개되어 있어요.

> 🔑 **핵심 개념**: `server_configs`에 `env` 키를 추가하면 해당 서버 프로세스에만 환경변수를 전달할 수 있어요. API 키를 서버별로 분리해서 관리하는 데 유용해요.

In [17]:
# ---------------------------------------------------
# Smithery 레지스트리 서버 구성 예시 (실행 불필요)
# ---------------------------------------------------
# 이 코드는 실제 실행하지 않아도 돼요
# Smithery에서 찾은 서버를 어떻게 설정하는지 패턴을 보여줘요

from pathlib import Path

# 노트북 실행 위치가 프로젝트 루트이든 챕터 폴더이든 동작하도록 프로젝트 루트를 찾아요
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "05_Agent_Development").exists()
)
WORKSPACE_DIR = PROJECT_ROOT / "05_Agent_Development" / "tmp" / "workspace"
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

# Smithery 스타일의 서버 구성 예시
smithery_examples = {
    # GitHub 서버: 저장소, PR, 이슈 관리
    "github": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-github"],
        "transport": "stdio",
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": "<your-token>"},  # 서버별 환경변수
    },
    # Filesystem 서버: 파일 읽기/쓰기
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(WORKSPACE_DIR)],
        "transport": "stdio",
    },
    # Context7 (이미 사용한 서버)
    "context7-mcp": {
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp@latest"],
        "transport": "stdio",
    },
}

def display_arg(arg: str) -> str:
    """프로젝트 내부 경로는 출력에서 상대 경로로 보여줘요."""
    path = Path(arg)
    if path.is_absolute():
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            pass
    return arg

# Smithery 서버 구성 예시:
for server_name, config in smithery_examples.items():
    cmd = " ".join([config["command"]] + [display_arg(arg) for arg in config["args"]])
    print(f"  {server_name}: {cmd}")
    
# 실제 사용 시: setup_mcp_tools(smithery_examples)

  github: npx -y @modelcontextprotocol/server-github
  filesystem: npx -y @modelcontextprotocol/server-filesystem 05_Agent_Development/tmp/workspace
  context7-mcp: npx -y @upstash/context7-mcp@latest


## 10. 전체 파이프라인: 4개 서버 통합

지금까지 배운 내용을 종합하여 **4개 서버를 통합한 완성형 에이전트**를 만들어볼게요.

| 서버 | 도구 | 전송 방식 |
|------|------|----------|
| 날씨 서버 | `get_weather` | stdio |
| 시간 서버 | `get_current_time` | HTTP |
| Context7 | `resolve-library-id`, `query-docs` | npx+stdio |
| 웹 검색 | `web_search`, `search_news` | stdio |

> 🎯 **강의 포인트**: 이 패턴이 실제 프로덕션에서 MCP를 활용하는 방식이에요. 여러 서버의 도구를 통합하면 에이전트가 **단계별 복합 작업**을 자율적으로 수행할 수 있어요.

In [18]:
# ---------------------------------------------------
# 4개 서버 통합 최종 파이프라인
# ---------------------------------------------------
full_server_configs = {
    # 1. 날씨 서버 (stdio)
    "weather": {
        "command": "uv",
        "args": ["run", "python", "servers/01_weather_server.py"],
        "transport": "stdio",
    },
    # 2. 시간 서버 (HTTP - 사전에 서버 실행 필요)
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",
        "transport": "streamable_http",
    },
    # 3. Context7 MCP (npx)
    "context7-mcp": {
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp@latest"],
        "transport": "stdio",
    },
    # 4. 웹 검색 서버 (실습 파일)
    "web_search": {
        "command": "uv",
        "args": ["run", "python", "servers/06_web_search_server.py"],
        "transport": "stdio",
    },
}

# HTTP 서버 연결 실패를 대비해서 예외 처리를 적용해요
try:
    all_tools = run_async(setup_mcp_tools(full_server_configs))
    print(f"\n4개 서버에서 총 {len(all_tools)}개의 도구를 로드했어요")
except Exception as e:
    print(f"경고: 4개 서버 통합 로드 실패 - {type(e).__name__}: {str(e)[:100]}")
    # 시간 서버(HTTP)가 실행 중이지 않을 수 있어요.
    # 별도 터미널에서 실행: uv run python servers/02_time_server.py
    
    # HTTP 서버 없이 stdio 서버만으로 재시도해요
    fallback_full_configs = {
        "weather": {
            "command": "uv",
            "args": ["run", "python", "servers/01_weather_server.py"],
            "transport": "stdio",
        },
        "context7-mcp": {
            "command": "npx",
            "args": ["-y", "@upstash/context7-mcp@latest"],
            "transport": "stdio",
        },
        "web_search": {
            "command": "uv",
            "args": ["run", "python", "servers/06_web_search_server.py"],
            "transport": "stdio",
        },
    }
    all_tools = run_async(setup_mcp_tools(fallback_full_configs))
    print(f"\n{len(all_tools)}개의 도구를 3개 서버(날씨 + Context7 + 웹검색)에서 로드했어요")

[MCP] 총 6개의 도구가 로드되었어요:
  - get_weather: 지정한 도시의 현재 날씨를 조회해요.

Args:
    location: 날씨를 조회할 도시 이름 (예: ...
  - get_current_time: 지정한 시간대의 현재 시간을 조회해요.

Args:
    timezone: 조회할 시간대 (기본값: Asi...
  - resolve-library-id: Resolves a package/product name to a Context7-compatible lib...
  - query-docs: Retrieves and queries up-to-date documentation and code exam...
  - web_search: Tavily API를 사용하여 실시간 웹 검색을 수행해요.

인터넷에서 최신 정보를 검색하고 결과를 요약하여...
  - search_news: 특정 주제에 대한 최신 뉴스를 검색해요.

지정된 주제에 대해 최근 며칠 이내의 뉴스를 검색하고
가장 관련성...

4개 서버에서 총 6개의 도구를 로드했어요


In [19]:
# ---------------------------------------------------
# 최종 통합 에이전트 실행
# ---------------------------------------------------
# 6개 도구를 모두 사용할 수 있는 에이전트
final_agent = create_agent(
    llm,
    all_tools,
    checkpointer=InMemorySaver(),
)

config = make_config()

# 복합 작업: 여러 서버를 순차적으로 활용
# 최종 복합 작업 실행 중...
run_async(run_agent_stream(
    final_agent,
    "다음 단계로 작업해줘: "
    "1) LangGraph 공식 문서에서 ToolNode 사용법을 검색해, "
    "2) 현재 시간과 서울 날씨를 확인해, "
    "3) 검색한 내용과 날씨/시간 정보를 종합해서 "
    "ToolNode를 사용하는 간단한 예제 코드를 작성해줘",
    config,
))


노드: model
================================== Ai Message ==================================
Tool Calls:
  resolve-library-id (call_Rwf0slNnoOkqVuc6vfUV1z36)
 Call ID: call_Rwf0slNnoOkqVuc6vfUV1z36
  Args:
    query: ToolNode 사용법
    libraryName: LangGraph

노드: tools
================================= Tool Message =================================
Name: resolve-library-id

[{'type': 'text', 'text': 'Available Libraries:\n\n- Title: LangGraph\n- Context7-compatible library ID: /langchain-ai/langgraph\n- Description: Build resilient language agents as graphs.\n- Code Snippets: 538\n- Source Reputation: High\n- Benchmark Score: 82.8\n- Versions: 0.2.74, 0.4.8, 0.5.3, 0.6.0, v0_0_8, 0_6_7, 1.0.3, prebuilt__1.0.4, 1.0.6, 1.0.8\n----------\n- Title: LangGraph\n- Context7-compatible library ID: /websites/langchain_oss_python_langgraph\n- Description: LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents, offering durable

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Context7 MCP 서버**: `npx @upstash/context7-mcp`로 최신 라이브러리 공식 문서를 실시간으로 검색할 수 있어요. LLM의 학습 데이터 한계를 극복하는 데 유용해요
- **3rd Party MCP 생태계**: Smithery AI 레지스트리에서 수백 개의 MCP 서버를 검색하고, `npx` 명령으로 설치 없이 바로 사용할 수 있어요
- **다중 서버 통합**: `MultiServerMCPClient` 딕셔너리에 서버를 추가하기만 하면 도구가 자동으로 통합돼요. `stdio`, `HTTP`, `npx` 방식 모두 동일하게 처리해요
- **복합 워크플로우**: MCP 도구(Context7, 웹 검색)와 일반 LangChain 도구(Tavily)를 리스트에 합쳐서 단일 에이전트에서 사용할 수 있어요
- **커스텀 워크플로우 경계**: `StateGraph + ToolNode` 구현은 07장 소유로 두고, 이 장에서는 Context7·다중 서버·도구 큐레이션 같은 고급 MCP 활용에 집중했어요

---

## 11. 도구 큐레이션

### 왜 도구를 줄여야 하는가

MCP 서버를 여러 개 연결하면 편리하지만, **도구 개수가 늘어날수록 에이전트 성능이 저하**될 수 있어요. MCP는 단순 연결 기술이 아니라, 에이전트에게 어떤 외부 행동을 허용할지 정하는 **하네스 인터페이스**이기도 합니다.

#### 하네스 관점에서 보는 MCP

- **Constrain**: 지금 단계에 필요한 도구만 노출해 선택지를 줄여요
- **Inform**: 도구 이름·description·schema로 모델에게 사용법을 알려줘요
- **HITL**: 외부 시스템을 변경하는 고위험 도구는 승인 절차를 붙여요

#### 모델이 많은 도구를 어려워하는 이유

여러 에이전트 설계 가이드와 산업 사례에서 공통적으로 지적하는 문제는 같아요. 모델에 제공되는 도구 수가 증가할수록 **도구 선택 정확도(tool selection accuracy)** 가 떨어질 수 있습니다. 모델이 어떤 도구를 언제 써야 할지 혼동하는 현상이 생기기 때문이에요.

주요 원인은 두 가지예요:

- **컨텍스트 비용**: 도구 정의(이름 + 설명 + 파라미터 스키마)는 모두 토큰을 소비해요. 도구가 많으면 실제 대화에 쓸 수 있는 컨텍스트 창이 줄어들어요.
- **선택 혼동**: LLM이 유사한 도구들을 구분하지 못하거나 불필요한 도구를 호출해 불필요한 API 비용과 지연이 발생해요.

#### 산업 사례: Harness.io MCP 서버 v2

DevOps 플랫폼 Harness.io는 MCP 서버 v1에서 **130개 이상의 도구**를 노출했어요. 이를 v2에서 **11개의 고수준 도구**로 재설계한 결과:

| 지표 | v1 (130+ 도구) | v2 (11 도구) |
|------|----------------|--------------|
| 컨텍스트 사용 비율 | ~26% | ~1.6% |
| 컨텍스트 창 기준 | 200K 토큰 | 200K 토큰 |
| 도구 선택 오류 | 높음 | 낮음 |

200K 토큰 컨텍스트 창에서 도구 정의만으로 26%를 소비하던 것이 1.6%로 줄었어요. 나머지 컨텍스트를 실제 대화와 추론에 더 많이 활용할 수 있게 됐어요.

> 참고: [Harness.io — MCP Server Redesign](https://www.harness.io/blog/harness-mcp-server-redesign)

#### 설계 원칙

- 도구 하나가 **단일 명확한 책임**을 갖도록 설계
- 비슷한 도구는 **하나의 도구 + 파라미터**로 통합
- 에이전트 워크플로의 각 단계에 **필요한 도구만** 노출

In [21]:
# ---------------------------------------------------
# 현재 노트북의 모든 MCP 도구 개수 및 컨텍스트 비용 측정
# ---------------------------------------------------
# 앞서 full_server_configs(4개 서버)로 로드한 all_tools를 재사용해요.
# 서버가 실행 중이지 않으면 context7_tools 또는 multi_tools를 사용해요.

# 측정할 도구 목록 선택 (이 노트북에서 이미 로드된 도구 사용)
try:
    tools_to_measure = all_tools        # 4개 서버 통합 도구
    source_label = "4개 서버 통합"
except NameError:
    try:
        tools_to_measure = multi_tools  # 3개 서버 통합 도구
        source_label = "3개 서버 통합"
    except NameError:
        tools_to_measure = context7_tools  # Context7 단독
        source_label = "Context7 단독"

print(f"[측정 대상] {source_label}")
print(f"전체 도구 개수: {len(tools_to_measure)}")
print()

# 도구별 이름·설명 길이 출력
for t in tools_to_measure:
    desc_len = len(t.description)
    print(f"  - {t.name}: {desc_len} 자")

# 총 도구 정의 길이 추정
# 실제 LLM 전송 시 도구 정의 = 이름 + 설명 + args_schema(JSON)
total_chars = sum(
    len(t.name) + len(t.description) + len(str(t.args_schema))
    for t in tools_to_measure
)
# 영문 기준 1 토큰 ≈ 4자, 한국어 포함 시 실제 토큰은 더 많을 수 있어요
estimated_tokens = total_chars // 4

print()
print(f"총 도구 정의 길이: ~{total_chars:,} 자")
print(f"추정 토큰 수:       ~{estimated_tokens:,} 토큰")
print()

# Harness 사례와 비교 (200K 컨텍스트 창 기준)
context_window = 200_000
usage_pct = (estimated_tokens / context_window) * 100
print(f"200K 컨텍스트 창 대비 도구 정의 비율: {usage_pct:.2f}%")
print(f"  (Harness v1 기준: ~26%,  v2 기준: ~1.6%)")

[측정 대상] 4개 서버 통합
전체 도구 개수: 6

  - get_weather: 109 자
  - get_current_time: 190 자
  - resolve-library-id: 2006 자
  - query-docs: 429 자
  - web_search: 315 자
  - search_news: 248 자

총 도구 정의 길이: ~5,817 자
추정 토큰 수:       ~1,454 토큰

200K 컨텍스트 창 대비 도구 정의 비율: 0.73%
  (Harness v1 기준: ~26%,  v2 기준: ~1.6%)


### 큐레이션 전략 3가지

도구 개수를 줄이는 방법은 크게 세 가지예요.

#### 1. 화이트리스트 (Whitelist)

필요한 도구 이름을 명시적으로 지정하고, 나머지는 에이전트에 노출하지 않아요.

```python
# MultiServerMCPClient가 반환한 전체 도구 중 필요한 것만 선택
needed = {"search_docs", "get_weather", "read_file"}
curated_tools = [t for t in tools if t.name in needed]
```

- 장점: 가장 단순하고 예측 가능
- 적합한 상황: 도구 이름을 미리 알고 있는 고정 워크플로

#### 2. 블랙리스트 (Blacklist)

특정 패턴의 도구를 제외하고 나머지를 허용해요.

```python
import re

# "debug_" 로 시작하는 내부 디버그 도구를 제외
curated_tools = [t for t in tools if not re.match(r"debug_", t.name)]
```

- 장점: 새 도구가 추가돼도 자동 포함
- 적합한 상황: 특정 범주(내부 전용, 위험 도구)만 차단하고 싶을 때

#### 3. 워크플로 기반 동적 선택 (Dynamic per Step)

StateGraph의 **각 노드(단계)마다 서로 다른 도구 부분집합**을 LLM에 바인딩해요.

```python
# 검색 단계: 검색 도구만 노출
search_llm = llm.bind_tools([search_tool, docs_tool])

# 생성 단계: 파일 저장 도구만 노출
write_llm  = llm.bind_tools([write_file_tool])
```

- 장점: 각 단계에서 도구 선택 혼동이 최소화
- 적합한 상황: 단계가 명확하게 구분된 파이프라인

> 실무에서는 세 전략을 혼합해요. 예를 들어 블랙리스트로 위험 도구를 먼저 제거하고, 단계별로 화이트리스트를 적용해요.

In [22]:
# ---------------------------------------------------
# 화이트리스트로 도구 부분집합 추출
# ---------------------------------------------------
# tools_to_measure 는 앞 셀에서 정의된 변수예요.
# 실제 노출할 도구 이름을 명시적으로 지정해요.

# 이 노트북에서 실제로 필요한 핵심 도구만 선택해요
# (서버마다 제공하는 실제 도구 이름을 먼저 위에서 확인하세요)
all_tool_names = {t.name for t in tools_to_measure}
print(f"사용 가능한 전체 도구 이름: {sorted(all_tool_names)}\n")

# 화이트리스트: 워크플로에 꼭 필요한 도구만 명시
# 실제 도구 이름에 맞게 조정하세요
NEEDED_TOOLS: set[str] = {
    "resolve-library-id",   # Context7: 라이브러리 ID 조회
    "get-library-docs",     # Context7: 문서 검색 (query-docs 대신 사용될 수 있어요)
    "get_weather",          # 날씨 서버
    "web_search",           # 웹 검색 서버
}

# 실제로 존재하는 도구만 필터링 (이름 불일치 방지)
curated_tools = [t for t in tools_to_measure if t.name in NEEDED_TOOLS]

print(f"전체 도구 개수:       {len(tools_to_measure)} 개")
print(f"큐레이션 후 도구 개수: {len(curated_tools)} 개")
print()

if curated_tools:
    # 큐레이션된 도구 목록:
    for t in curated_tools:
        print(f"  - {t.name}")
else:
    # 화이트리스트와 실제 도구 이름이 다를 때의 안전 처리
    # 참고: 화이트리스트와 실제 로드된 도구 이름이 다를 수 있어요.
    # 위에서 출력된 '사용 가능한 전체 도구 이름'을 NEEDED_TOOLS에 맞게 수정하세요.
    # 이 경우 처음 3개 도구를 샘플로 사용해요
    curated_tools = tools_to_measure[:3]
    print(f"대신 처음 {len(curated_tools)}개 도구를 샘플로 사용해요:")
    for t in curated_tools:
        print(f"  - {t.name}")

# 큐레이션 후 컨텍스트 비용 재측정
curated_chars = sum(
    len(t.name) + len(t.description) + len(str(t.args_schema))
    for t in curated_tools
)
curated_tokens = curated_chars // 4
curated_pct = (curated_tokens / 200_000) * 100

print()
print(f"큐레이션 후 추정 토큰: ~{curated_tokens:,} 토큰 ({curated_pct:.2f}% / 200K)")

사용 가능한 전체 도구 이름: ['get_current_time', 'get_weather', 'query-docs', 'resolve-library-id', 'search_news', 'web_search']

전체 도구 개수:       6 개
큐레이션 후 도구 개수: 3 개

  - get_weather
  - resolve-library-id
  - web_search

큐레이션 후 추정 토큰: ~906 토큰 (0.45% / 200K)


In [23]:
# ---------------------------------------------------
# 큐레이션된 도구로 에이전트 생성 및 간단한 latency 비교
# ---------------------------------------------------
import time

from langchain.agents import create_agent

QUESTION = "서울 날씨를 알려줘"

# --- 전체 도구 에이전트 ---
full_agent = create_agent(
    llm,
    tools_to_measure,
    checkpointer=InMemorySaver(),
)

async def measure_agent(agent, question: str, label: str) -> float:
    """에이전트 실행 시간을 측정해요."""
    config = make_config()
    inputs = {"messages": [("human", question)]}
    t0 = time.perf_counter()
    result = await agent.ainvoke(inputs, config)
    elapsed = time.perf_counter() - t0
    last_msg = result["messages"][-1]
    print(f"\n[{label}]")
    print(f"  도구 개수: {len(agent.nodes['tools'].tools_by_name)} 개"  # ToolNode 내부 도구 수
          if hasattr(agent.nodes.get("tools", None), "tools_by_name")
          else f"  도구 개수: (create_agent 내부)")
    print(f"  소요 시간: {elapsed:.2f}초")
    print(f"  응답 미리보기: {str(last_msg.content)[:120]}...")
    return elapsed

print(f"질문: '{QUESTION}'")
# ==================================================

# 전체 도구 에이전트 실행
elapsed_full = run_async(measure_agent(full_agent, QUESTION, f"전체 도구 ({len(tools_to_measure)}개)"))

# --- 큐레이션 도구 에이전트 ---
curated_agent = create_agent(
    llm,
    curated_tools,
    checkpointer=InMemorySaver(),
)

elapsed_curated = run_async(measure_agent(curated_agent, QUESTION, f"큐레이션 도구 ({len(curated_tools)}개)"))

# --- 비교 요약 ---
# ==================================================
# 비교 요약
print(f"  전체 도구:    {len(tools_to_measure)}개 / {elapsed_full:.2f}초")
print(f"  큐레이션 도구: {len(curated_tools)}개 / {elapsed_curated:.2f}초")
if elapsed_full > 0:
    diff_pct = (elapsed_full - elapsed_curated) / elapsed_full * 100
    print(f"  latency 개선: {diff_pct:+.1f}%")
print()
# 참고: latency는 네트워크 상태와 모델 부하에 따라 달라질 수 있어요.
#       컨텍스트 비용 절감 효과는 장기 대화에서 더욱 두드러져요.

질문: '서울 날씨를 알려줘'

[전체 도구 (6개)]
  도구 개수: (create_agent 내부)
  소요 시간: 2.57초
  응답 미리보기: 서울의 날씨는 맑습니다....

[큐레이션 도구 (3개)]
  도구 개수: (create_agent 내부)
  소요 시간: 2.57초
  응답 미리보기: 서울의 날씨는 맑고 화창합니다....
  전체 도구:    6개 / 2.57초
  큐레이션 도구: 3개 / 2.57초
  latency 개선: -0.0%



### 워크플로 단계별 동적 큐레이션 패턴 (개념)

실제 프로덕션 에이전트에서는 단일 도구 집합을 고정하는 것보다 **StateGraph 노드마다 필요한 도구만 바인딩**하는 방식이 효과적이에요.

#### 노드별 도구 바인딩 패턴

```python
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# 단계별 도구 분리
SEARCH_TOOLS = [resolve_library_id, get_library_docs, web_search]  # 검색 단계
WRITE_TOOLS   = [write_file, format_code]                           # 생성 단계

# 단계별로 서로 다른 LLM 인스턴스를 바인딩해요
search_llm = llm.bind_tools(SEARCH_TOOLS)
write_llm  = llm.bind_tools(WRITE_TOOLS)

async def search_node(state):
    """검색 도구만 보이는 LLM으로 정보를 수집해요."""
    return {"messages": [await search_llm.ainvoke(state["messages"])]}

async def write_node(state):
    """생성 도구만 보이는 LLM으로 결과를 작성해요."""
    return {"messages": [await write_llm.ainvoke(state["messages"])]}

graph = StateGraph(AgentState)
graph.add_node("search", search_node)
graph.add_node("search_tools", ToolNode(SEARCH_TOOLS))
graph.add_node("write", write_node)
graph.add_node("write_tools", ToolNode(WRITE_TOOLS))

graph.add_edge(START, "search")
graph.add_conditional_edges("search", tools_condition, {"tools": "search_tools", END: "write"})
graph.add_edge("search_tools", "search")
graph.add_conditional_edges("write", tools_condition, {"tools": "write_tools", END: END})
graph.add_edge("write_tools", "write")
```

#### 미들웨어로 단계별 필터링

Part 06에서 배울 `wrap_model_call` 미들웨어를 활용하면, 상태(state)에 현재 단계를 저장하고 모델 호출 시점에 동적으로 도구를 교체할 수 있어요.

```python
from langchain_core.runnables import RunnableConfig

def tool_filter_middleware(model_call):
    async def wrapped(state, config: RunnableConfig):
        phase = state.get("phase", "search")
        tools = SEARCH_TOOLS if phase == "search" else WRITE_TOOLS
        # 단계에 맞는 도구로 교체하고 호출
        return await llm.bind_tools(tools).ainvoke(state["messages"])
    return wrapped
```

> 이 패턴의 핵심은 **LLM이 한 번에 보는 도구 수를 최소화**하는 것이에요. 검색 단계에서는 생성 도구가, 생성 단계에서는 검색 도구가 전혀 보이지 않아요.

### 체크리스트 — MCP 서버 통합 전 점검

MCP 서버를 프로덕션 에이전트에 연결하기 전, 아래 항목을 반드시 확인하세요.

#### 도구 수 및 품질

- [ ] 에이전트에 노출되는 도구 개수가 **가급적 20개 이하**인가?
  - `20개`는 공식 한계가 아니라 강의용 경험칙이에요. 실제 기준은 작업 난이도, 모델, 도구 설명 품질에 따라 달라집니다.
  - 20개를 초과하면 먼저 단계별 바인딩, 화이트리스트, 고수준 도구 통합을 검토하세요.
- [ ] 각 도구의 `description`이 **200자 이내**로 간결한가?
  - 불필요하게 긴 설명은 토큰을 낭비하고 모델 혼동을 유발해요.
- [ ] 비슷한 기능의 도구가 **중복**되지 않는가?
  - 예: `search_web`과 `web_search`가 동시에 존재하면 모델이 혼동해요.
- [ ] 도구 이름이 **역할을 명확하게** 표현하는가?
  - 모델은 이름과 설명을 보고 도구를 선택해요. 모호한 이름은 오선택을 유발해요.

#### 워크플로 설계

- [ ] 에이전트 워크플로의 **각 단계마다 어떤 도구가 필요한지** 매핑했는가?
  - 단계별 도구 맵을 문서화하면 큐레이션 기준이 명확해져요.
- [ ] 단계별로 **서로 다른 도구 부분집합**을 바인딩하도록 설계했는가?
- [ ] 도구 선택 오류 발생 시 **폴백(fallback) 로직**이 있는가?

#### 권한 및 승인

- [ ] 읽기 전용 도구와 쓰기/삭제/결제/전송 도구를 구분했는가?
  - 읽기 도구는 자동 실행할 수 있어도, 외부 상태를 바꾸는 도구는 별도 승인 정책이 필요해요.
- [ ] 고위험 MCP 도구에 대해 **confirm / approve / reject** 같은 HITL 흐름을 설계했는가?
- [ ] 도구 description에 "언제 쓰면 안 되는지"가 포함되어 있는가?

#### 비용 및 모니터링

- [ ] `get_tools()`로 로드되는 도구 정의의 **총 토큰 추정치**를 측정했는가?
  - 위 측정 셀을 실행하여 컨텍스트 창 대비 비율을 확인하세요.
- [ ] LangSmith 추적으로 **도구 호출 패턴**을 모니터링하고 있는가?
  - 불필요한 도구 호출이 반복되면 큐레이션을 재검토해야 해요.
- [ ] MCP 서버 버전 업데이트 시 **도구 목록 변화**를 자동으로 감지하는가?

---

> 핵심 원칙: "모든 도구를 연결하고 싶은 충동을 참아라."
> 에이전트에게 필요한 도구만 보여주는 것이 성능과 비용 모두를 개선하는 가장 빠른 방법이에요.

## 다음 노트북 예고

다음 `Part 06 / 01-Middleware-Basics.ipynb`에서는 **LangChain V1 미들웨어**를 배워요. `create_agent`의 실행 흐름에 있는 정해진 후크(hook) 지점에 로직을 끼워 넣어 로깅·수정·제어·강제 기능을 구현하는 방법을 다룹니다. 핵심 후크 6가지(`before_agent`, `before_model`, `after_model`, `after_agent`, `wrap_model_call`, `wrap_tool_call`), 편의 데코레이터 `dynamic_prompt`, 대표 내장 미들웨어 6종을 통해 지금까지 만든 에이전트를 **안전하고 관측 가능한 프로덕션 에이전트**로 업그레이드하는 흐름을 배워요.